In [0]:
from pyspark.sql.functions import col, sha2, concat_ws, lit

# 1. Odczytujemy dane z tabeli Bronze (która czyta bezpośrednio z ADLS Gen2)
bronze_hr_df = spark.read.table("dbw_showcase.default.hr_bronze")

# 2. Generujemy Surrogate Key (SHA-2) - Wymóg z rozmowy rekrutacyjnej!
silver_hr_stg = bronze_hr_df.withColumn(
    "emp_sk", sha2(concat_ws("-", col("emp_id"), col("updated_at")), 256)
)

# Tworzymy tymczasowy widok, aby użyć go w zapytaniu SQL MERGE
silver_hr_stg.createOrReplaceTempView("stg_hr_updates")

# 3. Inicjalizacja tabeli Silver w dedykowanym kontenerze 'silver' na ADLS Gen2
spark.sql("""
CREATE TABLE IF NOT EXISTS dbw_showcase.default.hr_silver (
    emp_sk STRING,
    emp_id INT,
    emp_name STRING,
    city STRING,
    dept_id INT,
    department_name STRING,
    valid_from TIMESTAMP,
    valid_to TIMESTAMP,
    is_current BOOLEAN
)
USING DELTA
LOCATION 'abfss://silver@adlsportfolioaw2026.dfs.core.windows.net/tables/hr_silver'
""")

# 4. Proces MERGE INTO (SCD Type 2)
print("⏳ Uruchamiam proces MERGE INTO dla HR (SCD Type 2)...")

spark.sql("""
MERGE INTO dbw_showcase.default.hr_silver AS target
USING (
    SELECT 
        emp_sk, emp_id, emp_name, city, dept_id, department_name, 
        CAST(updated_at AS TIMESTAMP) AS valid_from 
    FROM stg_hr_updates
) AS source
ON target.emp_id = source.emp_id AND target.is_current = true

-- Kiedy pracownik istnieje, ale zmieniły się jego dane (np. przeprowadził się lub zmienił dział):
WHEN MATCHED AND (target.city <> source.city OR target.department_name <> source.department_name) THEN
  UPDATE SET 
    target.is_current = false, 
    target.valid_to = source.valid_from

-- Kiedy to zupełnie nowy pracownik lub nowy wpis historyczny:
WHEN NOT MATCHED THEN
  INSERT (emp_sk, emp_id, emp_name, city, dept_id, department_name, valid_from, valid_to, is_current)
  VALUES (source.emp_sk, source.emp_id, source.emp_name, source.city, source.dept_id, source.department_name, source.valid_from, null, true)
""")

print("✅ Tabela Silver HR została pomyślnie zaktualizowana i zapisana w kontenerze Silver!")